# Proyecto ML 23-F - Notebook principal

Este notebook será el punto de entrada reproducible del proyecto.

## 1. Configuración y rutas relativas

In [4]:
from pathlib import Path
import sys
ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
print(ROOT)

/Users/gabrielrezola/TrabajoML/RepositoriosGitHub/Proyecto-Machine-Learning-UNAV---Bayesianos-de-los-Ca-dos


## 2. Carga o descarga de datos

In [5]:
from src import data_download, preprocessing, eda, temporal_analysis, nlp_features, graph_analysis, clustering, classification, utils

print("Módulos importados correctamente")

Módulos importados correctamente


In [6]:
from pathlib import Path

for folder in ["data/raw", "data/interim", "data/processed"]:
    path = ROOT / folder
    print(f"\n{folder}:")
    if path.exists():
        files = list(path.iterdir())
        if files:
            for f in files:
                print(" -", f.name)
        else:
            print(" vacío")
    else:
        print(" no existe")


data/raw:
 - .gitkeep

data/interim:
 - .gitkeep

data/processed:
 - .gitkeep


In [7]:
print((ROOT / "src" / "data_download.py").read_text())

# Ingesta/listado de documentos pendiente de implementar.



In [9]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pathlib import Path

MONCLOA_URL = "https://www.lamoncloa.gob.es/consejodeministros/paginas/desclasificacion-documentos-23f.aspx"

response = requests.get(MONCLOA_URL, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

rows = []

for link in soup.find_all("a", href=True):
    href = link["href"]
    text = link.get_text(" ", strip=True)

    if ".pdf" in href.lower():
        full_url = urljoin(MONCLOA_URL, href)
        file_name = full_url.split("/")[-1]

        rows.append({
            "doc_id": f"DOC_{len(rows)+1:04d}",
            "title": text if text else file_name,
            "url": full_url,
            "file_name": file_name,
            "source": "La Moncloa"
        })

df_index = pd.DataFrame(rows)

output_path = ROOT / "data" / "interim" / "documents_index.csv"
df_index.to_csv(output_path, index=False)

print(f"Documentos encontrados: {len(df_index)}")
print(f"Archivo guardado en: {output_path}")

df_index.head()

Documentos encontrados: 156
Archivo guardado en: /Users/gabrielrezola/TrabajoML/RepositoriosGitHub/Proyecto-Machine-Learning-UNAV---Bayesianos-de-los-Ca-dos/data/interim/documents_index.csv


,doc_id,title,url,file_name,source
0,DOC_0001,BOE,https://www.boe.es/boe/dias/2026/02/25/pdfs/BO...,BOE-A-2026-4351.pdf,La Moncloa
1,DOC_0002,Transcripción de conversación telefónica de (p...,https://www.lamoncloa.gob.es/consejodeministro...,23F_1._Conversacion_telefonica_GARCIA_CARRES_y...,La Moncloa
2,DOC_0003,Transcripción de conversación telefónica de Ga...,https://www.lamoncloa.gob.es/consejodeministro...,23F_2._Conversacion_telefonica_GARCIA_CARRES.pdf,La Moncloa
3,DOC_0004,Conversaciones telefónicas de (presuntamente) ...,https://www.lamoncloa.gob.es/consejodeministro...,23F_3._Conversaciones_telefonicas_unidad_milit...,La Moncloa
4,DOC_0005,Documentación con una presunta planificación d...,https://www.lamoncloa.gob.es/consejodeministro...,23F_4._Documento_planificacion_del_golpe.pdf,La Moncloa


In [10]:
print("Filas y columnas:", df_index.shape)

print("\nDuplicados por URL:")
print(df_index["url"].duplicated().sum())

print("\nDuplicados por nombre de archivo:")
print(df_index["file_name"].duplicated().sum())

print("\nFuentes:")
print(df_index["source"].value_counts())

print("\nPrimeros 10 documentos:")
display(df_index.head(10))

Filas y columnas: (156, 5)

Duplicados por URL:
0

Duplicados por nombre de archivo:
0

Fuentes:
source
La Moncloa    156
Name: count, dtype: int64

Primeros 10 documentos:


,doc_id,title,url,file_name,source
0,DOC_0001,BOE,https://www.boe.es/boe/dias/2026/02/25/pdfs/BO...,BOE-A-2026-4351.pdf,La Moncloa
1,DOC_0002,Transcripción de conversación telefónica de (p...,https://www.lamoncloa.gob.es/consejodeministro...,23F_1._Conversacion_telefonica_GARCIA_CARRES_y...,La Moncloa
2,DOC_0003,Transcripción de conversación telefónica de Ga...,https://www.lamoncloa.gob.es/consejodeministro...,23F_2._Conversacion_telefonica_GARCIA_CARRES.pdf,La Moncloa
3,DOC_0004,Conversaciones telefónicas de (presuntamente) ...,https://www.lamoncloa.gob.es/consejodeministro...,23F_3._Conversaciones_telefonicas_unidad_milit...,La Moncloa
4,DOC_0005,Documentación con una presunta planificación d...,https://www.lamoncloa.gob.es/consejodeministro...,23F_4._Documento_planificacion_del_golpe.pdf,La Moncloa
5,DOC_0006,Documento manuscrito de posible planificación ...,https://www.lamoncloa.gob.es/consejodeministro...,23F_5._Documento_manuscrito_planificacion_del_...,La Moncloa
6,DOC_0007,Transcripción de cintas grabadas con conversac...,https://www.lamoncloa.gob.es/consejodeministro...,23F6TR_1.PDF,La Moncloa
7,DOC_0008,Nota del EM de la Guardia Civil con una secuen...,https://www.lamoncloa.gob.es/consejodeministro...,23F_7._Notas_Informativas_2_Seccion_EM_desarro...,La Moncloa
8,DOC_0009,Télex interiores y de agencias recibidos en 2ª...,https://www.lamoncloa.gob.es/consejodeministro...,23F_8._Telex_interiores_y_de_Agencias_recibido...,La Moncloa
9,DOC_0010,Oficio zona País Vasco que expresa una comunic...,https://www.lamoncloa.gob.es/consejodeministro...,23F_9._Oficio_dimanante_Zona_del_Pais_Vasco_di...,La Moncloa


In [11]:
import requests
from pathlib import Path

pdf_dir = ROOT / "data" / "raw" / "pdfs"
pdf_dir.mkdir(parents=True, exist_ok=True)

test_df = df_index.head(5)

for _, row in test_df.iterrows():
    url = row["url"]
    file_path = pdf_dir / row["file_name"]

    print(f"Descargando {row['doc_id']}: {row['file_name']}")

    response = requests.get(url, timeout=60)
    response.raise_for_status()

    file_path.write_bytes(response.content)

print("\nDescarga de prueba completada.")
print("Archivos guardados en:", pdf_dir)
print("Número de PDFs descargados:", len(list(pdf_dir.glob('*.pdf'))))

Descargando DOC_0001: BOE-A-2026-4351.pdf
Descargando DOC_0002: 23F_1._Conversacion_telefonica_GARCIA_CARRES_y_Tcol._TEJERO.pdf
Descargando DOC_0003: 23F_2._Conversacion_telefonica_GARCIA_CARRES.pdf
Descargando DOC_0004: 23F_3._Conversaciones_telefonicas_unidad_militar_El_Pardo.pdf
Descargando DOC_0005: 23F_4._Documento_planificacion_del_golpe.pdf

Descarga de prueba completada.
Archivos guardados en: /Users/gabrielrezola/TrabajoML/RepositoriosGitHub/Proyecto-Machine-Learning-UNAV---Bayesianos-de-los-Ca-dos/data/raw/pdfs
Número de PDFs descargados: 5


In [12]:
import pdfplumber
import pandas as pd

sample_rows = []

for _, row in test_df.iterrows():
    file_path = pdf_dir / row["file_name"]

    print(f"Extrayendo texto de {row['doc_id']}: {row['file_name']}")

    try:
        pages_text = []

        with pdfplumber.open(file_path) as pdf:
            n_pages = len(pdf.pages)

            for page in pdf.pages:
                text = page.extract_text() or ""
                pages_text.append(text)

        full_text = "\n".join(pages_text).strip()

        sample_rows.append({
            "doc_id": row["doc_id"],
            "title": row["title"],
            "file_name": row["file_name"],
            "file_path": str(file_path.relative_to(ROOT)),
            "n_pages": n_pages,
            "text_raw": full_text,
            "n_chars": len(full_text),
            "extraction_ok": len(full_text) > 0,
            "error": ""
        })

    except Exception as e:
        sample_rows.append({
            "doc_id": row["doc_id"],
            "title": row["title"],
            "file_name": row["file_name"],
            "file_path": str(file_path.relative_to(ROOT)),
            "n_pages": None,
            "text_raw": "",
            "n_chars": 0,
            "extraction_ok": False,
            "error": str(e)
        })

df_sample_text = pd.DataFrame(sample_rows)

output_path = ROOT / "data" / "interim" / "sample_text_extraction.csv"
df_sample_text.to_csv(output_path, index=False)

print("\nExtracción de prueba completada.")
print("Archivo guardado en:", output_path)
display(df_sample_text[["doc_id", "file_name", "n_pages", "n_chars", "extraction_ok", "error"]])

Extrayendo texto de DOC_0001: BOE-A-2026-4351.pdf
Extrayendo texto de DOC_0002: 23F_1._Conversacion_telefonica_GARCIA_CARRES_y_Tcol._TEJERO.pdf
Extrayendo texto de DOC_0003: 23F_2._Conversacion_telefonica_GARCIA_CARRES.pdf
Extrayendo texto de DOC_0004: 23F_3._Conversaciones_telefonicas_unidad_militar_El_Pardo.pdf
Extrayendo texto de DOC_0005: 23F_4._Documento_planificacion_del_golpe.pdf

Extracción de prueba completada.
Archivo guardado en: /Users/gabrielrezola/TrabajoML/RepositoriosGitHub/Proyecto-Machine-Learning-UNAV---Bayesianos-de-los-Ca-dos/data/interim/sample_text_extraction.csv


,doc_id,file_name,n_pages,n_chars,extraction_ok,error
0,DOC_0001,BOE-A-2026-4351.pdf,3,12359,True,
1,DOC_0002,23F_1._Conversacion_telefonica_GARCIA_CARRES_y...,4,0,False,
2,DOC_0003,23F_2._Conversacion_telefonica_GARCIA_CARRES.pdf,2,0,False,
3,DOC_0004,23F_3._Conversaciones_telefonicas_unidad_milit...,12,0,False,
4,DOC_0005,23F_4._Documento_planificacion_del_golpe.pdf,23,0,False,


In [13]:
import requests
from bs4 import BeautifulSoup

RTVE_URL = "https://www.rtve.es/noticias/23f-desclasificados/buscador-23f/"

response = requests.get(RTVE_URL, timeout=30)
print("Status code:", response.status_code)
print("Longitud HTML:", len(response.text))

soup = BeautifulSoup(response.text, "html.parser")

print("Título de la página:")
print(soup.title.get_text(strip=True) if soup.title else "Sin título")

print("\nPrimeros enlaces encontrados:")
for link in soup.find_all("a", href=True)[:20]:
    print(link.get_text(" ", strip=True)[:80], "→", link["href"])

Status code: 200
Longitud HTML: 42402
Título de la página:
Buscador 23F: explora los documentos desclasificados

Primeros enlaces encontrados:
Saltar al contenido principal → #mainContent
Saltar al menú de contenido → #nearNavmenu
Saltar al pie de página → #mainFooter
Volver a navegación principal → #mainNavmenu
Atentado Trump → https://www.rtve.es/noticias/20260426/servicio-secreto-evacua-a-trump-cena-corresponsales-tras-aparentes-disparos/17041533.shtml
Lieja Bastoña Lieja 2026 → https://www.rtve.es/deportes/20260426/lieja-bastogne-lieja-2026-directo-hoy-carrera-ciclismo-triptico-ardenas-ganador-resultado/17039640.shtml
Álex Márquez → https://www.rtve.es/deportes/20260425/gp-espana-jerez-2026-directo-carrera-motogp-resumen-domingo/17039503.shtml
Guerra en Irán → https://www.rtve.es/noticias/20260426/guerra-iran-ultima-hora-directo/17041402.shtml
Chernóbil 1986 → https://www.rtve.es/noticias/20260426/robots-humanos-limpiaron-desastre-nuclear-chernobil/17040844.shtml
Osasuna Sevilla → 

In [14]:
html = response.text

keywords = [
    "23F",
    "23-f",
    "desclasificados",
    "documentos",
    "pdf",
    "transcripcion",
    "transcripción",
    "resumen",
    "keywords",
    "json",
    "api"
]

for kw in keywords:
    count = html.lower().count(kw.lower())
    print(f"{kw}: {count}")

23F: 40
23-f: 0
desclasificados: 13
documentos: 8
pdf: 0
transcripcion: 0
transcripción: 0
resumen: 3
keywords: 0
json: 2
api: 1


## 3. Preprocesamiento textual

In [ ]:
# TODO: limpieza, tokenización, lematización y cálculo de features


## 4. EDA documental

In [ ]:
# TODO: distribución por fuente, fechas, longitud, calidad OCR


## 5. Mini caso 1 - Clasificación automática

In [ ]:
# TODO: baseline, TF-IDF, modelos y métricas


## 6. Mini caso 2 - Clustering y topics

In [ ]:
# TODO: embeddings, clustering, UMAP/BERTopic


## 7. Mini caso 3 - Grafo de entidades

In [ ]:
# TODO: NER, coocurrencias, networkx, centralidad


## 8. Mini caso 4 - Análisis temporal y buscador semántico

In [ ]:
# TODO: timeline, embeddings y recuperación semántica


## 9. Exportación de resultados

In [ ]:
# TODO: guardar figuras, tablas y datasets procesados
